# Profitability and business economics — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## Setup — caricamento e controlli

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'axes.titleweight':'semibold', 'axes.labelsize':10, 'font.size':10})
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.max_rows', 80)

candidates = [
    Path.cwd() / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent.parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd() / 'adidas_sales_features.csv',
    Path.cwd().parent / 'adidas_sales_features.csv',
]
file_input = next((p for p in candidates if p.is_file()), None)
if file_input is None:
    raise FileNotFoundError('Manca output/adidas_sales_features.csv. Esegui i notebook 01 e 02 del progetto, poi avvia questo notebook dalla root o dalla cartella notebooks/.')
df = pd.read_csv(file_input, parse_dates=['invoice_date'], dtype={'retailer_id': 'string'})
expected = {'retailer','invoice_date','region','state','city','product','price_per_unit',
            'units_sold','total_sales','operating_profit','operating_margin','sales_method',
            'year','month_number','month_name_it','year_month','weekday_number',
            'weekday_name_it','is_weekend','season_it','product_gender','product_category'}
assert expected.issubset(df.columns), f'Colonne mancanti: {expected-set(df.columns)}'
assert not df['invoice_date'].isna().any(), 'Date mancanti'
assert np.allclose(df['total_sales'], df['price_per_unit']*df['units_sold'],atol=.011), 'Ricavi non coerenti'
assert np.allclose(df['operating_profit'],df['total_sales']*df['operating_margin'],atol=.011), 'Profitto non coerente'

EURO_MONTH=['Gennaio','Febbraio','Marzo','Aprile','Maggio','Giugno','Luglio','Agosto','Settembre','Ottobre','Novembre','Dicembre']
DAYS=['Lunedì','Martedì','Mercoledì','Giovedì','Venerdì','Sabato','Domenica']
SEASONS=['Inverno','Primavera','Estate','Autunno']

def units(x): return f'{x:,.0f}'.replace(',', '.')
def usd(x): return f'${x:,.2f}' if abs(x)<1000 else f'${x:,.0f}'
def percent(x): return 'N/D' if pd.isna(x) else f'{x:,.2f}%'
def safe(a,b): return np.nan if b==0 or pd.isna(b) else a/b
def avg(x): return f'{x:,.2f}'
def note(message): display(Markdown('**Risultato:** '+str(message)))
def caution(message): display(Markdown('**Limite interpretativo:** '+str(message)))
def data_table(frame, rows=80):
    display(frame.head(rows).reset_index(drop=True).round(2))
    if len(frame)>rows:
        display(Markdown(f'_Visualizzate {rows} righe su {len(frame)}. La tabella completa rimane disponibile nella variabile Python._'))

def summary(dims, data=None):
    frame=df if data is None else data
    group=frame.groupby(dims,dropna=False,observed=True)
    result=group.agg(units=('units_sold','sum'),revenue=('total_sales','sum'),
                     profit=('operating_profit','sum'),rows=('units_sold','size'),
                     price_mean=('price_per_unit','mean'),days=('invoice_date','nunique')).reset_index()
    result['margin_pct']=np.where(result.revenue!=0,100*result.profit/result.revenue,np.nan)
    result['realized_price']=np.where(result.units!=0,result.revenue/result.units,np.nan)
    result['profit_unit']=np.where(result.units!=0,result.profit/result.units,np.nan)
    return result

def dominance(frame,dims,category,measure='units'):
    x=frame.sort_values(dims+[measure,category],ascending=[True]*len(dims)+[False,True])
    first=x.drop_duplicates(subset=dims,keep='first').copy()
    top=frame.groupby(dims)[measure].transform('max')
    tied=frame[frame[measure].eq(top)].groupby(dims,dropna=False)[category].nunique().reset_index(name='ex_aequo')
    return first.merge(tied,on=dims,how='left')

def bar(frame,label,value,title,top=12,ascending=False,ax=None):
    d=frame.sort_values(value,ascending=ascending).head(top).copy()
    if ax is None: fig,ax=plt.subplots(figsize=(11,max(4,0.40*len(d)+1.5)))
    else: fig=ax.figure
    d=d.iloc[::-1]
    ax.barh(d[label].astype(str),d[value]);ax.set_title(title);ax.set_xlabel(value.replace('_',' ').title());ax.grid(axis='x',alpha=.25)
    ax.tick_params(axis='y',labelsize=9)
    fig.tight_layout()
    return ax

def twobars(frame,label,metrics,titles,top=12):
    fig,axes=plt.subplots(1,2,figsize=(15,max(4,0.43*min(top,len(frame))+1)))
    for ax,m,t in zip(axes,metrics,titles):bar(frame,label,m,t,top=top,ax=ax)
    plt.show()

def stack(data,index,columns,measure,title,top=14,shares=True):
    g=data.pivot_table(index=index,columns=columns,values=measure,aggfunc='sum',fill_value=0)
    if g.empty:
        note('Non sono presenti dati per questo confronto.');return
    g=g.loc[g.sum(axis=1).sort_values(ascending=False).head(top).index]
    if shares: g=g.div(g.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    ax=g.plot(kind='bar',stacked=True,figsize=(max(10, min(16,len(g)*0.7+5)),5.5),width=.80)
    ax.set_title(title);ax.set_ylabel('Quota sul volume (%)' if shares else measure)
    ax.set_xlabel('');ax.legend(title=columns,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

def heat(data,index,columns,value,title,top=25,normalize=False):
    piv=data.pivot_table(index=index,columns=columns,values=value,aggfunc='sum',fill_value=0)
    if piv.empty: return
    piv=piv.loc[piv.sum(axis=1).sort_values(ascending=False).head(top).index]
    if normalize: piv=piv.div(piv.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    fig,ax=plt.subplots(figsize=(max(8,1.15*len(piv.columns)+4),max(4,.36*len(piv)+1.8)))
    im=ax.imshow(piv.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(np.arange(len(piv.columns)),piv.columns.astype(str),rotation=35,ha='right')
    ax.set_yticks(np.arange(len(piv.index)),piv.index.astype(str));ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.80,label='Quota (%)' if normalize else value)
    fig.tight_layout();plt.show()

def lineplot(frame,x,y,title,color_key=None):
    fig,ax=plt.subplots(figsize=(12,5))
    if color_key is None:
        t=frame.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3)
    else:
        for name,piece in frame.groupby(color_key,observed=True):
            t=piece.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3,label=str(name))
        ax.legend(title=color_key)
    ax.set_title(title);ax.set_ylabel(y);ax.tick_params(axis='x',rotation=70,labelsize=8)
    ax.grid(alpha=.25);fig.tight_layout();plt.show()

note(f'Caricate {units(len(df))} righe • periodo {df.invoice_date.min():%Y-%m-%d} — {df.invoice_date.max():%Y-%m-%d} • nessun CSV verrà esportato.')

## How do recorded and realized unit prices compare?

In [ ]:
parts=[]
for dims in [['year'],['retailer'],['product_category']]:
    t=summary(dims);t['gruppo']=' / '.join(dims);t['nome']=t[dims].astype(str).agg(' - '.join,axis=1)
    parts.append(t[['gruppo','nome','price_mean','realized_price','units']])
r=pd.concat(parts,ignore_index=True)
note(f"Prezzo medio semplice: {usd(df.price_per_unit.mean())}; prezzo medio realizzato (ricavi / unità): {usd(safe(df.total_sales.sum(),df.units_sold.sum()))}.")
t=r[r.gruppo=='product_category'].set_index('nome')[['price_mean','realized_price']]
t.plot(kind='bar',figsize=(9,5));plt.title('Prezzo semplice vs realizzato per categoria');plt.ylabel('USD/unità');plt.xticks(rotation=0);plt.tight_layout();plt.show()
data_table(r)

## How do volume metrics vary across product categories?

In [ ]:
p=summary(['product']);c=summary(['product_category']);g=summary(['product_gender'])
note(f"Unità complessive: {units(df.units_sold.sum())}; i tre raggruppamenti rappresentano gli stessi volumi, non vanno sommati tra loro.")
fig,axes=plt.subplots(1,3,figsize=(19,6));bar(p,'product','units','Prodotti',ax=axes[0]);bar(c,'product_category','units','Categorie',ax=axes[1]);bar(g,'product_gender','units','Target prodotto',ax=axes[2]);plt.show()
data_table(p);data_table(c);data_table(g)

## What is modeled total and annual revenue?

In [ ]:
t=summary(['year']);r=df.total_sales.sum();x=t.loc[t.year.eq(2020),'revenue'].iloc[0];y=t.loc[t.year.eq(2021),'revenue'].iloc[0]
note(f"Fatturato ricostruito complessivo: {usd(r)}; 2020: {usd(x)}; 2021: {usd(y)}. Il confronto è NON comparabile senza controllo di perimetro.")
bar(t.assign(year=t.year.astype(str)),'year','revenue','Ricavi ricostruiti per anno');plt.show()
data_table(t)

## What is modeled total and annual operating profit?

In [ ]:
t=summary(['year']);r=df.operating_profit.sum()
note(f"Profitto operativo ricostruito complessivo: {usd(r)}. Deriva dai ricavi rettificati × margini originari, non da costi industriali verificati.")
bar(t.assign(year=t.year.astype(str)),'year','profit','Profitti operativi ricostruiti per anno');plt.show()
data_table(t)

## How do weighted operating margins vary by segment?

In [ ]:
base=100*safe(df.operating_profit.sum(),df.total_sales.sum());dims=['year','retailer','product_category','sales_method','region']
note(f"Margine operativo complessivo ponderato sui ricavi: {percent(base)}. Nei sottogruppi si usa sempre SUM(profitto)/SUM(ricavi).")
fig,axes=plt.subplots(2,3,figsize=(19,11));axes=axes.ravel()
for k,dim in enumerate(dims):
    t=summary([dim]);t[dim]=t[dim].astype(str)
    bar(t,dim,'margin_pct','Margine (%) per '+dim,ax=axes[k])
axes[5].axis('off');plt.tight_layout();plt.show()
for dim in dims:data_table(summary([dim])[[dim,'units','revenue','profit','margin_pct']].sort_values('margin_pct',ascending=False))

## Which segments combine high volume with lower margins?

In [ ]:
t=summary(['retailer','product']);m_units=t.units.median();m_margin=t.margin_pct.median()
t['profilo']=np.select([(t.units>=m_units)&(t.margin_pct<m_margin),(t.units<m_units)&(t.margin_pct>=m_margin)],['Volume alto / margine basso','Volume basso / margine alto'],default='Altri')
a=t[t.profilo=='Volume alto / margine basso'].sort_values('units',ascending=False)
note(f"Su {len(t)} combinazioni retailer × prodotto: {len(a)} con volume ≥ mediana ({units(m_units)}) e margine < mediana ({percent(m_margin)}). Soglie esplorative, non target aziendali.")
fig,ax=plt.subplots(figsize=(11,6))
for name,part in t.groupby('profilo'):
    ax.scatter(part.units,part.margin_pct,label=name,s=50,alpha=.7)
ax.axvline(m_units,ls='--',color='gray');ax.axhline(m_margin,ls='--',color='gray')
ax.set_xlabel('Unità vendute');ax.set_ylabel('Margine (%)');ax.set_title('Retailer × prodotto: volume e redditività');ax.legend(loc='best');plt.tight_layout();plt.show()
data_table(t.sort_values(['profilo','units'],ascending=[True,False]))

## How concentrated is revenue across segments?

In [ ]:
parts=[]
for dim in ['retailer','product','state']:
    t=summary([dim]).sort_values('revenue',ascending=False).reset_index(drop=True)
    t['share_revenue_pct']=100*t.revenue/t.revenue.sum()
    t['cum_share_pct']=t.share_revenue_pct.cumsum()
    t['rank']=np.arange(1,len(t)+1);t['dimensione']=dim;t['nome']=t[dim].astype(str)
    parts.append(t[['dimensione','nome','rank','share_revenue_pct','cum_share_pct','revenue']])
r=pd.concat(parts,ignore_index=True)
note('; '.join(f"Top 3 {dim}: {percent(t.loc[t['rank']<=3,'share_revenue_pct'].sum())} del fatturato" for dim,t in r.groupby('dimensione'))+'. Quote INTERNE al dataset, non quote di mercato.')
fig,ax=plt.subplots(figsize=(10,6))
for dim,t in r.groupby('dimensione'):
    ax.plot(t['rank'],t['cum_share_pct'],marker='o',markersize=3,label=dim)
ax.set_xlabel('Top N segmenti');ax.set_ylabel('Quota cumulata fatturato (%)');ax.set_title('Concentrazione cumulata per dimensione');ax.legend();ax.set_ylim(0,103);plt.tight_layout();plt.show()
for dim,t in r.groupby('dimensione'):data_table(t)

## Which segments explain the observed cross-year difference?

In [ ]:
parts=[]
for dim in ['retailer','product_category','sales_method','region']:
    t=summary(['year',dim]);p=t.pivot(index=dim,columns='year',values='revenue').fillna(0)
    for y in [2020,2021]:
        if y not in p:p[y]=0
    p['delta_revenue']=p[2021]-p[2020];p['dimensione']=dim;p['segmento']=p.index.astype(str)
    parts.append(p.reset_index(drop=True))
r=pd.concat(parts,ignore_index=True);v=r.loc[r.delta_revenue.idxmax()]
note(f"Variazione maggiore nel confronto grezzo, {v.dimensione}: {v.segmento} ({usd(v.delta_revenue)}). Differenza NON attribuibile a crescita commerciale: le coperture annue sono diverse.")
fig,axes=plt.subplots(2,2,figsize=(17,11));axes=axes.ravel()
for ax,(dim,t) in zip(axes,r.groupby('dimensione')):
    bar(t,'segmento','delta_revenue','Ricavi 2021 meno 2020: '+dim,top=10,ax=ax)
plt.tight_layout();plt.show()
data_table(r.sort_values(['dimensione','delta_revenue'],ascending=[True,False]))
caution('Per valutare crescita a perimetro costante occorre selezionare almeno gli stessi stati/retailer presenti in entrambi gli anni e verificare continuità mensile.')